In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

# Load clean data
df = pd.read_csv('../data/processed/train_clean.csv')

# Features and target
feature_cols = [c for c in df.columns if c not in ['engine_id', 'cycle', 'RUL']]
X = df[feature_cols]
y = df['RUL']

print(f"Features: {len(feature_cols)}")
print(f"Samples: {len(X)}")

Features: 17
Samples: 20631


In [2]:
# Split by engine ID, not random rows
# Otherwise model sees future data from same engine
unique_engines = df['engine_id'].unique()
train_engines, test_engines = train_test_split(
    unique_engines, test_size=0.2, random_state=42
)

train_mask = df['engine_id'].isin(train_engines)
test_mask = df['engine_id'].isin(test_engines)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train engines: {len(train_engines)}")
print(f"Test engines: {len(test_engines)}")
print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Train engines: 80
Test engines: 20
Train samples: 16561
Test samples: 4070


In [4]:
import mlflow

# Use SQLite backend (works on all MLflow versions)
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("predictive_maintenance")

print("MLflow tracking started")

2026/07/08 15:54:00 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/08 15:54:00 INFO mlflow.store.db.utils: Updating database tables
2026/07/08 15:54:02 INFO mlflow.tracking.fluent: Experiment with name 'predictive_maintenance' does not exist. Creating a new experiment.


MLflow tracking started


In [5]:
with mlflow.start_run(run_name="random_forest_baseline"):
    # Parameters
    params = {
        "n_estimators": 100,
        "max_depth": 10,
        "random_state": 42
    }
    
    # Log params
    mlflow.log_params(params)
    
    # Train
    rf = RandomForestRegressor(**params)
    rf.fit(X_train, y_train)
    
    # Predict
    y_pred = rf.predict(X_test)
    
    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    
    # Log model
    mlflow.sklearn.log_model(rf, "model")
    
    print(f"Random Forest - RMSE: {rmse:.2f}, MAE: {mae:.2f}")

2026/07/08 15:55:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Random Forest - RMSE: 35.33, MAE: 25.88


In [7]:
with mlflow.start_run(run_name="xgboost_baseline"):
    params = {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "random_state": 42
    }
    
    mlflow.log_params(params)
    
    xgb_model = xgb.XGBRegressor(**params)
    xgb_model.fit(X_train, y_train)
    
    y_pred = xgb_model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    # Skip mlflow.sklearn.log_model for XGBoost
    # Save manually instead
    import pickle
    with open('../src/models/xgboost_model.pkl', 'wb') as f:
        pickle.dump(xgb_model, f)
    
    print(f"XGBoost - RMSE: {rmse:.2f}, MAE: {mae:.2f}")

XGBoost - RMSE: 35.54, MAE: 25.91


In [9]:
# Compare experiments using MLflow API instead of file paths
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("predictive_maintenance")
experiment_id = experiment.experiment_id

runs = client.search_runs(experiment_ids=[experiment_id])

print("All runs:")
for run in runs:
    rmse = run.data.metrics.get("rmse", "N/A")
    mae = run.data.metrics.get("mae", "N/A")
    print(f"  {run.info.run_name}: RMSE={rmse}, MAE={mae}")

# Find best
best_run = min(runs, key=lambda r: r.data.metrics.get("rmse", float('inf')))
best_run_id = best_run.info.run_id
print(f"\nBest run: {best_run.info.run_name} ({best_run_id})")
print(f"RMSE: {best_run.data.metrics['rmse']:.2f}")

All runs:
  xgboost_baseline: RMSE=35.54231426786423, MAE=25.905811309814453
  xgboost_baseline: RMSE=35.54231426786423, MAE=25.905811309814453
  random_forest_baseline: RMSE=35.32809125979548, MAE=25.877023242135113

Best run: random_forest_baseline (2c5e6bd6dec14e0387f94082c88aa6c2)
RMSE: 35.33


In [10]:
# Register best model
mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name="predictive_maintenance_model"
)
print("Model registered")

Successfully registered model 'predictive_maintenance_model'.
2026/07/08 16:53:52 WARNING mlflow.tracking._model_registry.fluent: Run with id 2c5e6bd6dec14e0387f94082c88aa6c2 has no artifacts at artifact path 'model', registering model based on models:/m-cde8e491cc3947bbbb5611f490cfe440 instead


Model registered


Created version '1' of model 'predictive_maintenance_model'.


In [11]:
import pickle

# Load best model from MLflow
best_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")

with open('../src/models/model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('../src/models/features.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("Model and features saved to src/models/")

Model and features saved to src/models/
